In [3]:


!pip install -q transformers datasets evaluate sacrebleu sentencepiece accelerate peft torch tqdm

import torch
from datasets import load_dataset
import evaluate
import numpy as np
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

# --------------------- CONFIG ---------------------
model_name = "facebook/nllb-200-distilled-600M"

# Update these paths after uploading your files
WORDS_CSV_PATH = "/content/marma_dictionary.csv"          # ← YOUR WORD-LEVEL CSV
SENTENCES_CSV_PATH = "/content/sentence_dataset.csv"  # ← YOUR SENTENCE-LEVEL CSV

# Same 8 directions as your previous code
directions = [
    ("English", "Marma_English_Transliterate", "eng_Latn"),
    ("Marma_English_Transliterate", "English", "eng_Latn"),
    ("Bangla", "Marma_English_Transliterate", "eng_Latn"),
    ("Marma_English_Transliterate", "Bangla", "ben_Beng"),
    ("Bangla", "Marma_Bangla_Transliterate", "ben_Beng"),
    ("Marma_Bangla_Transliterate", "Bangla", "ben_Beng"),
    ("English", "Marma_Bangla_Transliterate", "ben_Beng"),
    ("Marma_Bangla_Transliterate", "English", "eng_Latn"),
]

max_length = 128  # Small for words, fine for sentences too

# --------------------- TOKENIZER ---------------------
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
# We disable explicit lang tokens – transliterations will be treated by script/content

# --------------------- PREPROCESS FUNCTION ---------------------
def preprocess(examples, src_col, tgt_col):
    inputs = [str(x).strip() for x in examples[src_col]]
    targets = [str(x).strip() for x in examples[tgt_col]]
    model_inputs = tokenizer(inputs, max_length=max_length, truncation=True)
    model_inputs["labels"] = tokenizer(text_target=targets, max_length=max_length, truncation=True).input_ids
    return model_inputs

# --------------------- METRICS ---------------------
bleu = evaluate.load("sacrebleu")
wer = evaluate.load("wer")

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    if isinstance(preds, tuple):
        preds = preds[0]
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Strip whitespace for fair word-level comparison
    decoded_preds = [p.strip() for p in decoded_preds]
    decoded_labels = [l.strip() for l in decoded_labels]

    return {
        "exact_match": np.mean([p == l for p, l in zip(decoded_preds, decoded_labels)]),
        "bleu": bleu.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels])["score"],
        "wer": wer.compute(predictions=decoded_preds, references=decoded_labels),
    }

# --------------------- STAGE 1: TRAIN ON WORDS ---------------------
print("\n=== STAGE 1: Fine-tuning on WORD-LEVEL dataset ===\n")

raw_words = load_dataset("csv", data_files=WORDS_CSV_PATH)["train"]
raw_words = raw_words.filter(lambda x: all(isinstance(v, str) and v.strip() != "" for v in x.values()))
print(f"Word dataset loaded: {len(raw_words):,} word pairs")

word_models = {}  # To store best model dir per direction for stage 2

for idx, (src, tgt, lang_code) in enumerate(directions, 1):
    print(f"\n{'='*70}")
    print(f"WORD DIRECTION {idx}/8: {src} → {tgt}")
    print(f"{'='*70}")

    split = raw_words.train_test_split(test_size=0.1, seed=42)
    train_ds, val_ds = split["train"], split["test"]

    tokenized_train = train_ds.map(lambda x: preprocess(x, src, tgt), batched=True, remove_columns=raw_words.column_names)
    tokenized_val   = val_ds.map(lambda x: preprocess(x, src, tgt), batched=True, remove_columns=raw_words.column_names)

    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    model = get_peft_model(model, LoraConfig(
        task_type=TaskType.SEQ_2_SEQ_LM,
        r=64, lora_alpha=32, lora_dropout=0.1,
        target_modules=["q_proj", "v_proj"]
    ))
    model.print_trainable_parameters()

    args = Seq2SeqTrainingArguments(
        output_dir=f"./marma_word_{idx}",
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=3e-4,  # Slightly higher LR for word-level (faster alignment learning)
        per_device_train_batch_size=32,  # Larger batch – words are short
        per_device_eval_batch_size=32,
        weight_decay=0.01,
        num_train_epochs=15,  # More epochs possible on words (less overfitting risk per example)
        predict_with_generate=True,
        fp16=True,
        load_best_model_at_end=True,
        metric_for_best_model="exact_match",  # Prioritize exact word matches
        greater_is_better=True,
        report_to="none",
        logging_steps=10,
        save_total_limit=2,
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_val,
        tokenizer=tokenizer,
        data_collator=DataCollatorForSeq2Seq(tokenizer, padding=True),
        compute_metrics=compute_metrics,
    )

    trainer.train()

    save_dir = f"./MARMA_WORD_FINAL_{idx}_{src}_to_{tgt.replace(' ', '')}"
    trainer.save_model(save_dir)
    tokenizer.save_pretrained(save_dir)
    word_models[idx] = save_dir
    print(f"Word direction {idx} saved → {save_dir}")

    # Quick test on words
    print("\nQuick word test (3 examples):")
    test_examples = val_ds.select(range(min(3, len(val_ds))))
    test_texts = test_examples[src]
    inputs = tokenizer(test_texts, return_tensors="pt", padding=True).to("cuda" if torch.cuda.is_available() else "cpu")
    generated = model.generate(
        **inputs,
        max_new_tokens=32,
        forced_bos_token_id=tokenizer.lang_code_to_id.get(lang_code)  # Important for NLLB!
    )
    print("Input :", test_texts)
    print("Output:", tokenizer.batch_decode(generated, skip_special_tokens=True))

# --------------------- STAGE 2: CONTINUE ON SENTENCES ---------------------
print("\n=== STAGE 2: Continued fine-tuning on SENTENCE-LEVEL dataset ===\n")

raw_sentences = load_dataset("csv", data_files=SENTENCES_CSV_PATH)["train"]
raw_sentences = raw_sentences.filter(lambda x: all(isinstance(v, str) and v.strip() != "" for v in x.values()))
print(f"Sentence dataset loaded: {len(raw_sentences):,} sentences")

for idx, (src, tgt, lang_code) in enumerate(directions, 1):
    print(f"\n{'='*70}")
    print(f"SENTENCE DIRECTION {idx}/8: {src} → {tgt} (continuing from word model)")
    print(f"{'='*70}")

    split = raw_sentences.train_test_split(test_size=0.1, seed=42)
    train_ds, val_ds = split["train"], split["test"]

    tokenized_train = train_ds.map(lambda x: preprocess(x, src, tgt), batched=True, remove_columns=raw_sentences.column_names)
    tokenized_val   = val_ds.map(lambda x: preprocess(x, src, tgt), batched=True, remove_columns=raw_sentences.column_names)

    # Load the LoRA model from word-stage as starting point
    pretrained_dir = word_models[idx]
    model = AutoModelForSeq2SeqLM.from_pretrained(pretrained_dir)  # Loads base + LoRA

    args = Seq2SeqTrainingArguments(
        output_dir=f"./marma_sentence_{idx}",
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=1e-4,  # Lower LR to preserve word alignments
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        weight_decay=0.01,
        num_train_epochs=10,
        predict_with_generate=True,
        fp16=True,
        load_best_model_at_end=True,
        metric_for_best_model="bleu",  # Now focus on BLEU for sentences
        greater_is_better=True,
        report_to="none",
        logging_steps=20,
        save_total_limit=2,
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_val,
        tokenizer=tokenizer,
        data_collator=DataCollatorForSeq2Seq(tokenizer, padding=True),
        compute_metrics=compute_metrics,
    )

    trainer.train()

    final_save_dir = f"./MARMA_SENTENCE_FINAL_{idx}_{src}_to_{tgt.replace(' ', '')}"
    trainer.save_model(final_save_dir)
    tokenizer.save_pretrained(final_save_dir)
    print(f"Final sentence direction {idx} saved → {final_save_dir}")

    # Quick sentence test
    print("\nQuick sentence test (3 examples):")
    test_examples = val_ds.select(range(min(3, len(val_ds))))
    test_texts = test_examples[src]
    inputs = tokenizer(test_texts, return_tensors="pt", padding=True).to("cuda" if torch.cuda.is_available() else "cpu")
    generated = model.generate(
        **inputs,
        max_new_tokens=128,
        forced_bos_token_id=tokenizer.lang_code_to_id.get(lang_code)
    )
    print("Input :", test_texts)
    print("Output:", tokenizer.batch_decode(generated, skip_special_tokens=True))

print("\n=== ALL DONE! You now have 8 final sentence models initialized with strong word-level knowledge ===")

ImportError: To be able to use evaluate-metric/wer, you need to install the following dependencies['jiwer'] using 'pip install jiwer' for instance'

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q transformers datasets evaluate sacrebleu sentencepiece accelerate peft torch tqdm

import torch
from datasets import load_dataset
import evaluate
import numpy as np
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

# --------------------- CONFIG ---------------------
model_name = "facebook/nllb-200-distilled-600M"

# Update these paths after uploading your files
WORDS_CSV_PATH = "/content/marma_dictionary.csv"          # ← YOUR WORD-LEVEL CSV
SENTENCES_CSV_PATH = "/content/sentence_dataset.csv"  # ← YOUR SENTENCE-LEVEL CSV

# Same 8 directions as your previous code
directions = [
    ("English", "Marma_English_Transliterate", "eng_Latn"),
    ("Marma_English_Transliterate", "English", "eng_Latn"),
    ("Bangla", "Marma_English_Transliterate", "eng_Latn"),
    ("Marma_English_Transliterate", "Bangla", "ben_Beng"),
    ("Bangla", "Marma_Bangla_Transliterate", "ben_Beng"),
    ("Marma_Bangla_Transliterate", "Bangla", "ben_Beng"),
    ("English", "Marma_Bangla_Transliterate", "ben_Beng"),
    ("Marma_Bangla_Transliterate", "English", "eng_Latn"),
]

max_length = 128  # Small for words, fine for sentences too

# --------------------- TOKENIZER ---------------------
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
# We disable explicit lang tokens – transliterations will be treated by script/content

# --------------------- PREPROCESS FUNCTION ---------------------
def preprocess(examples, src_col, tgt_col):
    inputs = [str(x).strip() for x in examples[src_col]]
    targets = [str(x).strip() for x in examples[tgt_col]]
    model_inputs = tokenizer(inputs, max_length=max_length, truncation=True)
    model_inputs["labels"] = tokenizer(text_target=targets, max_length=max_length, truncation=True).input_ids
    return model_inputs

# --------------------- METRICS ---------------------
bleu = evaluate.load("sacrebleu")
wer = evaluate.load("wer")

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    if isinstance(preds, tuple):
        preds = preds[0]
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Strip whitespace for fair word-level comparison
    decoded_preds = [p.strip() for p in decoded_preds]
    decoded_labels = [l.strip() for l in decoded_labels]

    return {
        "exact_match": np.mean([p == l for p, l in zip(decoded_preds, decoded_labels)]),
        "bleu": bleu.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels])["score"],
        "wer": wer.compute(predictions=decoded_preds, references=decoded_labels),
    }


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 11.6 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

ImportError: To be able to use evaluate-metric/wer, you need to install the following dependencies['jiwer'] using 'pip install jiwer' for instance'